# Logon — RAW audit 

**Goal:** Understand the *true* shape of `logon.csv` directly from RAW before touching ETL/emit.

**What this notebook does**
1. Loads `data/<RELEASE>/logon.csv` via `notebooks/nb_paths.py`.
2. Prints schema, dtypes, nulls, value distributions.
3. Parses timestamps conservatively and inspects after-hours rates.
4. Checks Logon/Logoff alternation and missing-pair noise.
5. Profiles user↔PC usage; optionally uses LDAP (if present) to exclude admins for the shared-PC audit.

No Parquet is written here; this is a *viewer* notebook only.

In [ ]:
# === Cell 1 — Bootstrap & imports (robust to where you run the notebook) ===
import pandas as pd
from pathlib import Path
import sys

# Make repo root importable whether you're in notebooks/ or elsewhere
CWD = Path.cwd()
if (CWD / "notebooks").exists():
    sys.path.insert(0, str(CWD))              # running from repo root
else:
    sys.path.insert(0, str(CWD.parent))       # running from notebooks/

try:
    from notebooks.nb_paths import bootstrap
except Exception as e:
    # Last-resort: climb one more level if someone opened the ipynb from a nested folder
    sys.path.insert(0, str(CWD.parent.parent))
    from notebooks.nb_paths import bootstrap

# Optional: override release here (e.g., "r5.1")
RELEASE_OVERRIDE = None

env = bootstrap(RELEASE_OVERRIDE)
LOGON_PATH = env.RAW / "logon.csv"

print(f"RELEASE = {env.RELEASE}")
print(f"RAW dir = {env.RAW}")
print(f"LOGON   = {LOGON_PATH}  exists: {LOGON_PATH.exists()}")

if not LOGON_PATH.exists():
    raise FileNotFoundError(
        f"logon.csv not found at {LOGON_PATH}\n"
        f"Check release.txt (currently '{env.RELEASE}') or place the CSV at data/{env.RELEASE}/logon.csv"
    )

In [ ]:
# === Cell 2: Load raw (no schema coercion beyond dtype=str) ===
from notebooks.nb_paths import read_csv
raw = read_csv(env, "logon")  # dtype=str, on_bad_lines='warn'
print("Rows x Cols:", raw.shape)
print("Columns:", list(raw.columns))
display(raw.head(10))

In [ ]:
# === Cell 3: Schema + nulls + dtypes ===
def _nulls(df):
    s = df.isna().sum().sort_values(ascending=False)
    return pd.DataFrame({"nulls": s, "null_rate": (s/len(df)).round(4)})

display(pd.DataFrame(raw.dtypes, columns=["dtype"]))
display(_nulls(raw))

for col in ["date","user","pc","activity","id"]:
    if col in raw.columns:
        vc = raw[col].value_counts(dropna=False).head(10)
        print(f"\nTop values for {col}:")
        display(vc)

In [ ]:
# === Cell 4: Activity audit ===
if "activity" in raw.columns:
    acts = raw["activity"].astype(str).str.strip()
    print("Unique activities (raw):", sorted(acts.unique())[:50])
    print("Counts:")
    display(acts.value_counts().to_frame("count"))
else:
    print("[warn] No 'activity' column found in raw logon CSV.")

In [ ]:
# === Cell 5: Timestamp parsing sanity (from 'date') ===
if "date" not in raw.columns:
    raise AssertionError("Missing 'date' column in logon.csv — cannot proceed.")

ts_probe = pd.to_datetime(raw["date"], errors="coerce")
ok = ts_probe.notna().sum()
print(f"Parsed timestamps: {ok}/{len(ts_probe)} ({ok/len(ts_probe):.1%})")
if ok:
    print("Range:", ts_probe.min(), "→", ts_probe.max())
    print("Example hours:", ts_probe.dt.hour.dropna().value_counts().head(10).to_dict())

In [ ]:
# === Cell 6: Build a minimal 'clean' view (in-memory only) ===
from src.helpers.time import add_timestamp_and_month
from src.helpers.users import normalize_user_series 

clean = add_timestamp_and_month(raw.copy(), "date")
clean["user_key"] = normalize_user_series(clean.get("user", pd.Series(index=clean.index, dtype=object)))
if "activity" in clean.columns:
    clean["activity"] = clean["activity"].astype(str).str.strip().str.lower()

keep = ["timestamp","event_month","user_key","user","pc","activity","id","date"]
clean = clean[[c for c in keep if c in clean.columns]]
print("Rows:", len(clean))
print("Parsed-ts rate:", clean["timestamp"].notna().mean() if "timestamp" in clean.columns else None)
display(clean.head(10))

In [ ]:
# === Cell 7: User-level distributions ===
u = clean["user_key"].dropna() if "user_key" in clean.columns else pd.Series([], dtype=object)
print("unique users:", u.nunique())
evt_per_user = clean.groupby("user_key").size().sort_values(ascending=False)
display(evt_per_user.describe(percentiles=[0.5,0.9,0.95,0.99]))
print("Top 10 users by events:")
display(evt_per_user.head(10))

In [ ]:
# === Cell 8: PC-level distributions ===
if "pc" in clean.columns:
    c_pc = clean[clean["pc"].notna()]
    print("unique PCs:", c_pc["pc"].nunique())
    users_per_pc = c_pc.groupby("pc")["user_key"].nunique().sort_values(ascending=False)
    print("PCs by distinct users (top 15):")
    display(users_per_pc.head(15))
else:
    print("[warn] No 'pc' column present.")

In [ ]:
# === Cell 9: Logon/Logoff alternation probe ===
if {"user_key","timestamp","activity"}.issubset(clean.columns):
    # pick a few users with many events
    top_users = clean.groupby("user_key").size().sort_values(ascending=False).head(5).index.tolist()
    for uk in top_users:
        sub = clean.loc[clean["user_key"] == uk, ["timestamp","activity","pc"]].dropna().sort_values("timestamp")
        # count adjacent duplicates of 'logon' (suggests missing logoff noise)
        dup_logon = ((sub["activity"] == "logon") & (sub["activity"].shift(1) == "logon")).sum()
        dup_logoff = ((sub["activity"] == "logoff") & (sub["activity"].shift(1) == "logoff")).sum()
        print(f"User {uk}: rows={len(sub)} dup_logon_pairs={dup_logon} dup_logoff_pairs={dup_logoff}")
        display(sub.head(8))
else:
    print("[skip] Need user_key, timestamp, activity for alternation probe.")

In [ ]:
# === Cell 10: Daily coverage + weekday/weekend profile ===
if "timestamp" in clean.columns:
    ts = pd.to_datetime(clean["timestamp"], errors="coerce")
    days = ts.dt.date.value_counts().sort_index()
    print("First/last day:", days.index.min(), "→", days.index.max())
    print("Daily row-count quantiles:")
    display(days.describe())
    weekday = ts.dt.weekday.value_counts().sort_index()
    print("weekday distribution (0=Mon):", weekday.to_dict())
else:
    print("[skip] No timestamp column parsed.")

In [ ]:
# === Cell 11: After-hours rate (19:00–07:00) ===
if "timestamp" in clean.columns:
    ts = pd.to_datetime(clean["timestamp"], errors="coerce")
    ah = ts.dt.hour.lt(7) | ts.dt.hour.ge(19)
    rate = float(ah.mean()) if len(ah) else 0.0
    print(f"After-hours events: {ah.sum()}/{len(ah)} ({rate:.1%})")
    # per-user after-hours rate (top 10)
    by_user = clean.assign(_ah=ah).groupby("user_key")["_ah"].mean().sort_values(ascending=False).head(10)
    print("Top 10 users by AH rate:")
    display(by_user)
else:
    print("[skip] No timestamp column parsed.")

In [ ]:
# ETL: ensure LDAP exists, then build LOGON with this kernel's interpreter
import sys, shlex, subprocess, importlib, pathlib
from src.helpers.io import out_path
from notebooks.nb_paths import bootstrap

py = sys.executable
etl = importlib.import_module("scripts.etl")
repo_root = pathlib.Path(etl.__file__).resolve().parent.parent

# 0) Ensure LDAP as-of (lean) exists; build if missing
env = bootstrap()
ldap_asof_p = pathlib.Path(out_path(env, "ldap_v3_lean", "ldap_asof_by_month"))
if not ldap_asof_p.exists():
    cmd_ldap = f'{py} -m scripts.etl ldap --family ldap_v3 --profile lean --overwrite'
    print("> " + cmd_ldap)
    proc_ldap = subprocess.run(shlex.split(cmd_ldap), cwd=str(repo_root), capture_output=True, text=True)
    print(proc_ldap.stdout)
    if proc_ldap.returncode != 0:
        print(proc_ldap.stderr)
        raise RuntimeError(f"LDAP ETL failed with code {proc_ldap.returncode}")

# 1) Build LOGON (writes only two domain artifacts + two helper parquets)
cmd_logon = f'{py} -m scripts.etl logon --family logon_v3 --ldap-family-for-join ldap_v3_lean --overwrite'
print("> " + cmd_logon)
proc_logon = subprocess.run(shlex.split(cmd_logon), cwd=str(repo_root), capture_output=True, text=True)
print(proc_logon.stdout)
if proc_logon.returncode != 0:
    print(proc_logon.stderr)
    raise RuntimeError(f"logon ETL failed with code {proc_logon.returncode}")

In [ ]:
# === Final QC summary for LOGON artifacts (one cell) ===
from pathlib import Path
import json, pandas as pd
from notebooks.nb_paths import bootstrap
from src.helpers.io import out_dir, out_path

# Toggle strictness here:
STRICT = True           # True = fail fast with asserts; False = print warnings only
SHARED_MIN, SHARED_MAX = 95, 120
ASSIGNED_NULL_MAX = 0.02

env = bootstrap()

# artifact paths
base   = Path(out_dir(env, "logon_v3"))
full_p = base / "logon_v3_full" / "logon_full.parquet"
lean_p = base / "logon_v3_lean" / "logon_lean.parquet"

print(f"full exists: {full_p.exists()} | {full_p}")
print(f"lean exists: {lean_p.exists()} | {lean_p}")
if STRICT:
    assert full_p.exists() and lean_p.exists(), "Missing final logon parquets."

# helpers
shared_p   = Path(out_path(env, "logon_v3", "shared_pcs"))
assigned_p = Path(out_path(env, "logon_v3", "assigned_pc"))

sp = pd.read_parquet(shared_p)  if shared_p.exists()   else pd.DataFrame()
ap = pd.read_parquet(assigned_p) if assigned_p.exists() else pd.DataFrame()

# method from QC meta
qc_meta_p = Path(env.OUT) / env.RELEASE / "qc" / "shared_pc_candidates_meta.json"
method = "unknown"; chosen_K = None; approx = None
if qc_meta_p.exists():
    m = json.loads(qc_meta_p.read_text())
    chosen_K = m.get("chosen_K")
    # current ETL writes this key name:
    approx   = m.get("approx_count_PCs_ge_K_or_topN")
    method   = "threshold-K" if chosen_K is not None else "top-100"

# counts
if not sp.empty:
    if "pc" in sp.columns:
        mask = sp.get("shared_pc", True)
        shared_ct = int(sp.loc[mask == True, "pc"].nunique())
    else:
        shared_ct = int(len(sp))
else:
    shared_ct = 0

if not ap.empty and "assigned_pc" in ap.columns:
    null_rate = float(ap["assigned_pc"].isna().mean())
else:
    null_rate = 1.0

print(f"method: {method} | chosen_K: {chosen_K} | approx_count: {approx}")
print(f"shared PCs: {shared_ct}")
print(f"assigned_pc null rate: {null_rate:.1%}")

# checks
msg = []
if not (SHARED_MIN <= shared_ct <= SHARED_MAX):
    msg.append(f"shared_pc count {shared_ct} outside expected ~100 [{SHARED_MIN}..{SHARED_MAX}]")
if null_rate > ASSIGNED_NULL_MAX:
    msg.append(f"assigned_pc null rate too high: {null_rate:.2%} (>{ASSIGNED_NULL_MAX:.0%})")

if msg:
    text = " | ".join(msg)
    if STRICT:
        raise AssertionError(text)
    else:
        print("[warn]", text)

# tiny preview
cols = [c for c in ["user_key","assigned_pc","days_on_pc","user_days","share"] if c in ap.columns]
display(ap[cols].head(10) if cols else ap.head(10))

---
## What to decide after this audit
- Confirm `date/user/pc/activity` are clean and consistent.
- Confirm after-hours definition fits the data (19:00–07:00).
- Confirm missing-pair noise is small enough to tolerate.
- Sanity-check candidate shared PCs and that one-modal assigned PCs per user exist.

When you’re satisfied, move on to **ETL/emit** changes with confidence.